In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
df = pd.read_csv('Dataset/Complete_dataset.csv')
df.drop(columns=['Unnamed: 0'],inplace=True)
df_copy = df.copy()

In [3]:
df_copy

,date,PM1,PM2.5,PM10,TSP
0,2016-01-01,NaN,NaN,NaN,NaN
1,2016-01-02,NaN,NaN,NaN,NaN
2,2016-01-03,NaN,NaN,NaN,NaN
3,2016-01-04,NaN,NaN,NaN,NaN
4,2016-01-05,NaN,NaN,NaN,NaN
...,...,...,...,...,...
3226,2024-12-27,NaN,90.031286,122.154243,161.637471
3227,2024-12-28,NaN,115.462085,150.554242,191.303695
3228,2024-12-29,NaN,104.691319,139.022639,201.824028
3229,2024-12-30,NaN,81.418264,107.658472,139.620417


In [4]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3231 entries, 0 to 3230
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    3231 non-null   object 
 1   PM1     1310 non-null   float64
 2   PM2.5   2465 non-null   float64
 3   PM10    2450 non-null   float64
 4   TSP     2286 non-null   float64
dtypes: float64(4), object(1)
memory usage: 126.3+ KB


In [5]:
df_copy.sample(5)

,date,PM1,PM2.5,PM10,TSP
528,2017-06-12,17.31399,32.633910,108.746200,367.71700
1152,2019-02-26,19.37783,28.642060,52.607750,85.92163
2995,2024-05-10,NaN,32.197639,48.886389,NaN
2079,2021-09-18,NaN,18.502355,35.100000,94.70000
2219,2022-03-26,NaN,NaN,NaN,NaN


In [6]:
df_copy.isnull().sum()

date        0
PM1      1921
PM2.5     766
PM10      781
TSP       945
dtype: int64

In [7]:
df_copy.drop(columns=['PM1'],inplace=True)

In [8]:
df_copy['datetime'] = pd.to_datetime(df_copy['date'])
df_copy.set_index('datetime', inplace=True)

In [9]:
df_copy.drop(columns=['date'],inplace=True)

##### Interpolation for missing values.

##### This for linear `` y=y1+((x−x1)*(y2−y1)/(x2−x1)) ``
##### We are using time which will produce nearly same values.

In [10]:
# Create complete daily date range
full_dates = pd.date_range(
    start='2016-08-25',
    end='2024-12-31',
    freq='D'
)

# Find missing dates
missing_dates = full_dates.difference(df_copy.index)

# Total missing dates
print("Total missing dates:", len(missing_dates))

# Show missing dates
print(missing_dates)

Total missing dates: 57
DatetimeIndex(['2020-12-31', '2021-01-05', '2021-03-22', '2021-03-23',
               '2021-07-26', '2021-07-27', '2021-07-28', '2021-07-29',
               '2021-10-03', '2021-10-05', '2021-10-06', '2021-10-07',
               '2021-10-08', '2021-10-09', '2021-10-10', '2021-10-11',
               '2021-10-12', '2021-10-13', '2021-10-14', '2021-10-15',
               '2021-10-16', '2021-10-17', '2021-10-18', '2021-10-19',
               '2021-10-20', '2021-10-21', '2021-10-22', '2021-10-23',
               '2021-10-24', '2021-10-25', '2021-10-26', '2021-10-27',
               '2021-10-28', '2021-10-29', '2021-10-30', '2021-10-31',
               '2021-11-01', '2021-11-02', '2021-11-03', '2021-11-04',
               '2021-11-05', '2021-11-06', '2021-11-07', '2021-11-08',
               '2021-11-09', '2021-11-10', '2021-11-11', '2021-11-12',
               '2021-11-13', '2021-11-14', '2021-11-15', '2021-11-16',
               '2021-11-17', '2021-11-18', '2021-11-1

##### These are those rows where all elements (parameters are missing)

#### Code here creates a continuous daily time-series dataset by adding missing dates between the first and last observation. Missing pollutant values are then estimated using time-based interpolation for gaps of up to 14 consecutive days. Finally, any remaining missing values are removed to prepare a clean dataset for model training.

In [15]:
full_dates = pd.date_range(
    start=df_copy.index.min(),
    end=df_copy.index.max(),
    freq='D'
)

df_copy = df_copy.reindex(full_dates)
df_copy = df_copy.interpolate(method='time', limit=5)
df_copy = df_copy.dropna()

In [16]:
df_copy.isnull().sum()

PM2.5    0
PM10     0
TSP      0
dtype: int64

In [17]:
len(df_copy)

2646

In [18]:
df_copy.to_csv('cleaned_dataset.csv')

##### Removed unusable PM1 column.
##### Converted datetime properly.
##### Reindex missing calendar dates.
##### Used controlled interpolation (limit=14).
##### Removed unreliable long-gap regions.